## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. It also installs `pymc` and `arviz`, which this analysis needs, into the same session. You can read every line of the code below before you run it.

# New bottle-cap supplier: faulty rate against a 2% limit

**Bayesian Estimation of the Population Proportion** · a worked example from the StatNex teaching guide

**The story.** A drinks company switched to a new supplier of bottle caps. The first batch was checked cap by cap: 2 of 200 were faulty (1%). The contract says the faulty rate must stay below 2%. New suppliers have run at about 3% faulty in their first months, and the quality manager does not want to ignore that experience.

**The question.** How likely is it that the new supplier's true faulty rate is below the 2% limit?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

This analysis needs `pymc` and `arviz`. This cell installs them. If Colab asks you to restart the session, restart it and choose **Run all** again.

In [ ]:
%pip install -q "pymc>=6.3" "arviz>=1.3"

## The table

This cell saves the table `14_bayesian/bayes_proportion.csv` (2 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Variable,New supplier
Sample size,200.0
Defect rate,0.01
""")

## Libraries

In [ ]:
import arviz as az
import numpy as np
import pandas as pd
import pymc as pm

pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.6g}".format)

## The data

In [ ]:
# Each column holds the sample size and the proportion, top to bottom.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
variables = ["New supplier"]
credible = 95 / 100
comparison = 0.02

## The prior

In [ ]:
# Informative: a prior proportion m, trusted as much as w observations, is the beta
# distribution with mean m that adds w imaginary observations, Beta(m w, (1 - m) w).
prior_mean, prior_weight = 0.03, 100
a0, b0 = prior_mean * prior_weight, (1 - prior_mean) * prior_weight

## Posterior of the proportion: PyMC

In [ ]:
# The model: x successes in n independent trials, each with the same probability. A beta
# prior on a binomial count gives a beta posterior, Beta(a0 + x, b0 + n - x), so the
# update itself is two sums. PyMC samples that posterior by MCMC (the NUTS sampler),
# which is how it handles the models that have no such shortcut; with 100,000 draws its
# summaries agree with the exact ones to about two or three significant figures.
low, high = (1 - credible) / 2, (1 + credible) / 2
update, posterior, chance = {}, {}, {}
for name in variables:
    size, p = pd.to_numeric(data[name], errors="coerce").dropna().to_numpy(float)[:2]
    n = int(round(size))
    x = int(round(n * p))  # the number of successes
    with pm.Model():
        theta = pm.Beta("theta", alpha=a0, beta=b0)
        pm.Binomial("successes", n=n, p=theta, observed=x)
        trace = pm.sample(draws=25_000, tune=2_000, chains=4, random_seed=2026, progressbar=False)
        # The posterior mode (the MAP estimate), found by optimisation rather than by
        # sampling.
        mode = float(pm.find_MAP(progressbar=False)["theta"]) if a0 + x > 1 and b0 + n - x > 1 else np.nan
    print(f"\n{name}")
    print(az.summary(trace, kind="stats", ci_prob=credible, ci_kind="eti"))
    draws = trace.posterior["theta"].to_numpy().ravel()
    update[name] = {"n": n, "successes": x, "proportion": p, "prior mean": a0 / (a0 + b0),
                    "posterior alpha": a0 + x, "posterior beta": b0 + n - x}
    posterior[name] = {"mean": draws.mean(), "median": np.median(draws), "mode": mode,
                       "sd": draws.std(ddof=1), "lower": np.quantile(draws, low),
                       "upper": np.quantile(draws, high)}
    # The probability that the proportion is below (above) the comparison value is the
    # share of draws below (above) it.
    chance[name] = {"comparison": comparison, "P(below)": np.mean(draws < comparison),
                    "P(above)": np.mean(draws > comparison)}

update = pd.DataFrame(update)
posterior = pd.DataFrame(posterior)
print("\nFrom prior to posterior")
print(update)
print(f"\nPosterior of the proportion ({credible:.0%} credible interval)")
print(posterior)
chance = pd.DataFrame(chance)
print("\nProbability of being below or above the comparison value")
print(chance)

## What the result says

The batch had 2 faulty caps in 200, a sample proportion of 0.01. The prior is what was believed before the data: here Beta(3, 97), with a mean of 0.03, from the experience with earlier new suppliers. The posterior joins the prior and the data. Its mean is 0.0167, so the best estimate of the faulty rate is about 1.7%.

The 95% credible interval, the range that holds the true rate with 95% probability, runs from 0.005 to 0.034. That is from about 0.5% to 3.4%, so the 2% limit lies inside it.

The probability that the true rate is below 0.02 is 0.715, and the probability that it is above is 0.285. The supplier probably meets the contract, but this is not yet sure. More batches will narrow the answer.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.